# ⚙️ 02 — Preprocessing & Feature Engineering
**Project:** ISIC 2024 — Skin Cancer Detection

**Goal:** Build the image augmentation pipeline, engineer patient-relative tabular features (ugly duckling signal), create Patient-level Group K-Fold splits, and verify the PyTorch DataLoader.

---

## 📦 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import albumentations as A
from sklearn.model_selection import GroupKFold
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
IMG_DIR  = DATA_DIR / 'train-image' / 'image'
PROC_DIR.mkdir(exist_ok=True)

N_FOLDS  = 5
IMG_SIZE = 128
SEED     = 42

print('Ready.')

## 🔁 2. Image Augmentation Pipeline

In [ ]:
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MedianBlur(blur_limit=5, p=1.0),
    ], p=0.2),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, fill_value=0, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# Visualize augmentations on a malignant sample
df = pd.read_csv(DATA_DIR / 'train-metadata.csv')
mal_sample = df[df['target'] == 1].sample(1, random_state=42)['isic_id'].values[0]
sample_path = str(IMG_DIR / f'{mal_sample}.jpg')

if Path(sample_path).exists():
    original = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes[0, 0].imshow(original); axes[0, 0].set_title('Original', fontweight='bold'); axes[0, 0].axis('off')
    for i in range(1, 10):
        aug = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
            A.Rotate(limit=45, p=0.9),
            A.RandomBrightnessContrast(p=0.7),
        ])(image=original)['image']
        r, c = divmod(i, 5)
        axes[r, c].imshow(aug); axes[r, c].set_title(f'Aug #{i}'); axes[r, c].axis('off')
    plt.suptitle('Augmentation Examples (Malignant Lesion)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/figures/augmentation_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sample image not found — confirm data/raw/ path.')

## 🏗️ 3. Patient-Relative Feature Engineering (Ugly Duckling Signal)

Key insight from top solutions: A malignant lesion tends to stand out from a patient's other lesions.
We encode this by computing each lesion's deviation from its patient's baseline.

In [ ]:
def engineer_patient_relative_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute patient-level relative features (ugly duckling signal).
    For each numerical feature, compute each lesion's deviation from its patient's mean.
    """
    df = df.copy()

    # Features to compute patient-relative stats for
    rel_features = [
        'clin_size_long_diam_mm',   # lesion physical size
        'tbp_lv_areaMM2',           # lesion area
        'tbp_lv_norm_color',        # normalized color
        'tbp_lv_color_std_mean',    # color variation
        'tbp_lv_symm_2axis',        # asymmetry
        'tbp_lv_deltaLBnorm',       # color delta from background
    ]

    rel_features = [f for f in rel_features if f in df.columns]

    # Patient-level aggregations
    for feat in rel_features:
        patient_stats = df.groupby('patient_id')[feat].agg(['mean', 'std', 'min', 'max'])
        patient_stats.columns = [f'patient_{feat}_{s}' for s in ['mean', 'std', 'min', 'max']]
        df = df.merge(patient_stats, on='patient_id', how='left')

        # Deviation from patient mean (z-score)
        df[f'{feat}_vs_patient_mean'] = (
            df[feat] - df[f'patient_{feat}_mean']
        ) / (df[f'patient_{feat}_std'] + 1e-6)

        # Ratio to patient mean
        df[f'{feat}_ratio_to_patient'] = df[feat] / (df[f'patient_{feat}_mean'] + 1e-6)

    # Lesion count per patient (context feature)
    df['patient_lesion_count'] = df.groupby('patient_id')['patient_id'].transform('count')

    # Rank of lesion size within patient's lesions
    if 'clin_size_long_diam_mm' in df.columns:
        df['lesion_size_rank_in_patient'] = df.groupby('patient_id')['clin_size_long_diam_mm']\
            .rank(pct=True)

    print(f'Added {len([c for c in df.columns if "vs_patient" in c or "ratio_to" in c])} patient-relative features')
    return df


df_engineered = engineer_patient_relative_features(df)
print(f'Feature count: {df.shape[1]} → {df_engineered.shape[1]}')
print('\nSample new features:')
new_cols = [c for c in df_engineered.columns if 'vs_patient' in c or 'ratio_to' in c]
print(df_engineered[new_cols[:6]].describe().round(3))

### Validate: Do Patient-Relative Features Separate Classes?

In [ ]:
rel_feats_to_plot = [c for c in df_engineered.columns if 'vs_patient_mean' in c][:4]

if rel_feats_to_plot:
    fig, axes = plt.subplots(1, len(rel_feats_to_plot), figsize=(5*len(rel_feats_to_plot), 5))
    if len(rel_feats_to_plot) == 1:
        axes = [axes]

    for ax, feat in zip(axes, rel_feats_to_plot):
        b = df_engineered[df_engineered['target'] == 0][feat].dropna().clip(-5, 5)
        m = df_engineered[df_engineered['target'] == 1][feat].dropna().clip(-5, 5)
        ax.hist(b, bins=50, alpha=0.6, density=True, color='#2196F3', label='Benign')
        ax.hist(m, bins=20, alpha=0.8, density=True, color='#F44336', label='Malignant')
        ax.set_title(feat.replace('_vs_patient_mean', '\n(vs patient mean)'), fontsize=9, fontweight='bold')
        ax.legend(fontsize=9)

    plt.suptitle('Patient-Relative Features: Ugly Duckling Signal', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/figures/patient_relative_features.png', dpi=150, bbox_inches='tight')
    plt.show()

## ✂️ 4. Patient-Level Group K-Fold Split

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
df_engineered['fold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(
    gkf.split(df_engineered, df_engineered['target'], groups=df_engineered['patient_id'])
):
    df_engineered.loc[val_idx, 'fold'] = fold_idx

# Verify: no patient appears in both train and val
print(f'Fold assignments: {dict(df_engineered["fold"].value_counts().sort_index())}')
print()

for fold in range(N_FOLDS):
    val_fold  = df_engineered[df_engineered['fold'] == fold]
    train_fold = df_engineered[df_engineered['fold'] != fold]
    overlap    = set(val_fold['patient_id']) & set(train_fold['patient_id'])
    mal_val    = val_fold['target'].sum()
    print(f'Fold {fold}: train={len(train_fold):,}, val={len(val_fold):,}, '
          f'malignant in val={mal_val}, patient overlap={len(overlap)} (should be 0)')

df_engineered.to_csv(PROC_DIR / 'train_folds.csv', index=False)
print(f'\nSaved to: {PROC_DIR}/train_folds.csv')

## ✅ 5. Summary

| Step | Result |
|------|--------|
| Augmentation | 8 transforms incl. CoarseDropout for simulating lesion occlusion |
| Feature engineering | Added patient-relative z-scores, ratios, rank features |
| CV strategy | 5-fold Patient GroupKFold — zero patient leakage verified |
| Saved | `data/processed/train_folds.csv` with fold column |

**Next:** `03_Model_Training.ipynb` — train EfficientNetV2-S with dynamic undersampling, 5-fold CV.